In [ ]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
import jax_sn

import jax_sbgeom as jsb
import jax
import matplotlib.pyplot as plt
import numpy as onp
import equinox as eqx
jax.config.update("jax_enable_x64", True)
from jax_sbgeom.flux_surfaces import FluxSurfaceNormalExtendedNoPhi, ToroidalExtent, FluxSurface
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
vmec_file = ["/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4"][0]
fs_base = FluxSurfaceNormalExtendedNoPhi.from_hdf5(vmec_file)
from functools import partial
import pyvista as pv
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import lineax as lx
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver, MultiGroupOperatorScanCheckpoint
from jax_sn.solution_domain import SolutionDomainData
from typing import Tuple, Callable
from dataclasses import dataclass


In [ ]:
from breeding_blanket_setup import _get_bb_args, _setup_bb_mgop_symmetric, aux_data_dict, bb_data_dict

In [ ]:
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver, MultiGroupOperator
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import _solve_multigroup_operator_rhs_scan

from jax_sn.parametrisation.multigroup_parametrisation import MultigroupOperatorParametrisedVertices, MultigroupOperatorParametrisedObjective, MultigroupOperatorParametrisedVerticesNoToposort

from jax_sn.tools.parametrisation_tools import ParametrisedModule, ParametrisationValues, ParametrisedModuleTheta

In [ ]:
from jax_sn.operators.transport_operator import TransportOperatorVmapPallas, TransportOperatorMultipleAnglesVmap


In [ ]:
class BZShieldValues(ParametrisationValues):
    bzshield : jnp.ndarray

class BZShieldSettings(eqx.Module):
    resolution_layers : Tuple[int]
    base_layer_size   : jnp.ndarray
    
from breeding_blanket_setup import recreate_symmetric_vertices

@eqx.filter_jit
def create_bzshield_vertices(params : BZShieldValues, settings : BZShieldSettings):
    bzshield = params.bzshield
    resolution_layers = settings.resolution_layers
    base_layer_size   = settings.base_layer_size    
    size              =   base_layer_size.at[3].set(bzshield[0]).at[6].set(bzshield[1])    
    return recreate_symmetric_vertices(size, resolution_layers)[:, None]
    
class BZShieldParametrisation(ParametrisedModuleTheta):

    additional_settings : BZShieldSettings

    def __check_init__(self):
        assert isinstance(self.base_module, MultiGroupOperator), "Base module must be a MultiGroupOperator."
        assert isinstance(self.additional_settings, BZShieldSettings), "Additional settings must be of type BZShieldSettings."
    
    @eqx.filter_jit
    def to_module(self, theta : BZShieldValues):
        return MultigroupOperatorParametrisedVerticesNoToposort.reconstruct_operator(self.base_module, create_bzshield_vertices(theta, self.additional_settings))
     
def get_solution_domain(MGOP, element):
    return jax_sn.solution_domain.SolutionDomain(MGOP.transport_operator.transport_operator.solution_domain_data, element)

def fast_neutron_flux(solution_values, Ij, egs_fast):
    last_element = solution_values.shape[-2] - 1
    return jnp.einsum("ij, j->",solution_values[egs_fast, 0, last_element], Ij)    

def distance(theta : BZShieldValues, settings : BZShieldSettings):
    bzshield          = theta.bzshield    
    base_layer_size   = settings.base_layer_size    
    size              = base_layer_size.at[3].set(bzshield[0]).at[6].set(bzshield[1])    
    return jnp.sum(size[1:])

def tritium_breeding_ratio(solution_values, settings :BZShieldSettings, solution_domain_data : SolutionDomainData, tritium_xs):
    bz_elements = sum(settings.resolution_layers) + sum(settings.resolution_layers[1:3]) # sum all previous layers (including plasma) and then first two actual blanket layers        
    solution_values_bz = jax.lax.dynamic_slice(solution_values[:,0, :, : ], (0, bz_elements, 0 ), (solution_values.shape[0], settings.resolution_layers[3], solution_values.shape[-1])) #[n_energy_groups, n_elements_bz, n_basis_per_element]
    tritium_xs_bz  = tritium_xs['BreedingZone']['nt']
    jacobian_dets = jax.lax.dynamic_slice(solution_domain_data.precomputed_geometric_data.vol_jacobians_det, (bz_elements,), (settings.resolution_layers[3],))  #[n_elements_bz,]
    return jnp.einsum("g, geb, b, e -> ", tritium_xs_bz, solution_values_bz, solution_domain_data.fem_matrices.integration_coeff, jacobian_dets) * 1e2 / jnp.sqrt(4 * jnp.pi)

class BZShieldMultiGroupObjective(MultigroupOperatorParametrisedObjective):
    
    aux_data : dict

    def objective(self, theta : BZShieldValues):
        return 0    

    def tritium_breeding_ratio(self, theta : BZShieldValues):
        mgop_theta = self.theta_parametrised_module.to_module(theta)
        solution_values = self._to_solution(mgop_theta).value
        return tritium_breeding_ratio(solution_values, self.theta_parametrised_module.additional_settings, mgop_theta.transport_operator.transport_operator.solution_domain_data, self.aux_data)



In [ ]:
import mcbb
from mcbb.linear_blanket import OpenMCLinearBlanketSimulation, OpenMCSymmetricLinearBlanketSimulation
import openmc

In [ ]:
hcpb_blanket = mcbb.examples.hcpb()
hcpb_blanket.layers[0].thickness = 0.1
hcpb_blanket.plasma_boundary_type = 'reflective'

groups_list = [14.1e6, 1e6, 1e4, 300, 10,0]

In [ ]:
settings = mcbb.blanket_base.OpenMCSettingsBlanket(True, "openmc_data/", 10, 1e4)

In [ ]:
simulation_symm = OpenMCSymmetricLinearBlanketSimulation(hcpb_blanket, settings, 200, groups_list)

In [ ]:

mgxs_lib, tally_result_dict =simulation_symm.create_mgxs_library(onp.flip(groups_list), 6)


In [ ]:
mgxs_lib_new = mcbb.openmc_base_functions.mgxs_lib_to_standard_formatting(mgxs_lib)

In [ ]:
new_bb_data_dict = {key : (mgxs_lib_new[key]['total'], mgxs_lib_new[key]['nu-scatter matrix']) for key in bb_data_dict().keys()}
new_aux_data_dict = {key : {'nt' : mgxs_lib_new[key]['(n,Xt)']} for key in aux_data_dict().keys()}

In [ ]:
dim = 1 
degree = 7
dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order = _get_bb_args(dim, degree)

aux_data = aux_data_dict()
MGOP, source_total, solution_domain = _setup_bb_mgop_symmetric(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order, new_bb_data_dict)
pallas_op = TransportOperatorMultipleAnglesVmap(TransportOperatorVmapPallas(MGOP.transport_operator.transport_operator.solution_domain_data, MGOP.transport_operator.transport_operator.topological_sorted_angle, 0))
MGOPcp    = MultiGroupOperatorScanCheckpoint(MGOP.moment_operator, MGOP.discrete_operator, pallas_op, MGOP.mass_operator, MGOP.gather_scatter_operator, MGOP.n_energy_groups, MGOP.transpose_bool)


values_init              = BZShieldValues(bzshield = jnp.array([0.5, 0.2]))
additional_settings      = BZShieldSettings(tuple(n_elem_regions), jnp.array(d_elem_regions))
source_all_eg            = jnp.stack([source_total if i==0 else jnp.zeros_like(source_total) for i in range(MGOP.n_energy_groups)])

operator_parametrisation = BZShieldParametrisation(MGOPcp, additional_settings)
multigroup_objective     = BZShieldMultiGroupObjective(operator_parametrisation, source_all_eg, MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(1e-12,1e-12, max_steps= 100)), aux_data= new_aux_data_dict)
multigroup_objective.tritium_breeding_ratio(values_init)

In [ ]:
print(simulation_symm.geometry.get_cells_by_name("Plasma")[0].bounding_box)

In [ ]:
tally_result_dict['tritium']['mean'].shape

In [ ]:
tbr_openmc = jnp.sum(tally_result_dict['tritium']['mean'] * tally_result_dict['tally_volume'][:, None ])

In [ ]:
from breeding_blanket_setup import _get_solution_interp

x, solution_values = _get_solution_interp(get_solution_domain(multigroup_objective.to_module(values_init), solution_domain.element), multigroup_objective.solution(values_init).value)

In [ ]:

for i in range(solution_values.shape[0]):    
    plt.stairs(tally_result_dict['flux']['mean'][...,i], tally_result_dict['flux']['x'])
plt.yscale('log')
plt.plot(x * 100 - jnp.sum(jnp.array(d_elem_regions)[1:]) * 100 - 10, solution_values.T * 1e2 / onp.sqrt(4 * onp.pi))


In [ ]:
print(result_dict['flux']['mean'].shape)
for i in range(solution_values.shape[0]):    
    plt.stairs(result_dict['flux']['mean'][...,i], result_dict['flux']['x'])
plt.yscale('log')
plt.plot(x * 100, solution_values.T * 1e2 / onp.sqrt(4 * onp.pi))


In [ ]:
energy_groups = [14.1e6, 1e6, 300, 10, 1e-10]
energy_filter = openmc.EnergyFilter(onp.flip(energy_groups))

In [ ]:
source = hcpb_blanket.create_source(geometry, 1.0)
settings = mcbb.openmc_base_functions.create_openmc_settings(10, 1e5, source)
tallies = hcpb_blanket.create_tallies(onp.linspace(geometry.bounding_box[0][0], geometry.bounding_box[1][0], 50), energy_filter, geometry)

In [ ]:
multigroup_objective.objective(values_init)

In [ ]:
@eqx.filter_jit
def objective_function(base_obj : BZShieldMultiGroupObjective, theta : BZShieldValues):
    return base_obj.objective(theta)

dobj_dtheta = eqx.filter_jit(jax.grad(objective_function, argnums=1))

In [ ]:
multigroup_objective.objective(values_init)

In [ ]:
solution_domain = get_solution_domain(operator_parametrisation.to_module(values_init), solution_domain.element)
solution_value = multigroup_objective.solution(values_init)

In [ ]:
xmax = jnp.max(solution_domain.solution_domain_data.domain.vertices[:,0])
print(xmax)

In [ ]:
solution_domain.interpolate(jnp.array([[xmax]]), solution_value.value[0,0])

In [ ]:
import matplotlib.pyplot as plt
x = jnp.linspace(0, jnp.max(solution_domain.solution_domain_data.domain.vertices[:,0]), 100)[:, None]
solution_values = jnp.array([solution_domain.interpolate(x, value[i,0,...]) for i in range(value.shape[0])])
plt.semilogy(x, solution_values.T)
plt.xlabel("x")
plt.ylabel(r"$\phi$")


In [ ]:
from breeding_blanket_setup import plot_solution_values

plot_solution_values(get_solution_domain(operator_parametrisation.to_module(values_init), solution_domain.element), multigroup_objective.solution(values_init).value)

In [ ]:
print(bb_args_1D)

dim = 1
degree= 6
print(bb_args_1D)
region_sizes = [0.2,0.002,0.025,1.0,  0.385, 0.06, 0.25,0.06]
region_resolutions = bb_args_1D[3]
d_other_dim = 5.0
n_other_dim = 1.0
tn_order = 3

MGOP2, source_total2, solution_domain2 = _setup_bb_args(dim, degree, region_sizes, region_resolutions, d_other_dim, n_other_dim, tn_order)

In [ ]:
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import _solve_multigroup_operator_rhs_scan

result2 = _solve_multigroup_operator_rhs_scan(MGOP2, jnp.stack([source_total2 if i==0 else jnp.zeros_like(source_total2) for i in range(MGOP2.n_energy_groups)]), MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(1e-12,1e-12, max_steps= 100))).value

In [ ]:
onp.testing.assert_allclose(result2, solution_value)